In financial services and markets, instruments and securities have unique identifiers. These identifiers are essential to ensure that all parties involved in trading are referring to the same instrument. Products such as bonds, currencies, and structured derivatives each have their own unique identifiers and conventions for creating new ones. Common identifiers include Ticker, ISIN, CUSIP, SEDOL, RIC, BBGID, and others.

PermID(from RFT/LSEG) and OpenFigi(Blmberg) are initiatives from the respective companies to address this issue with unique identifiers. PermID and OpenFigi are open source initiatives that will allow other instruments to be mapped to them(PermID/OpenFigi).

OpenFigi: https://openfigi.com
Permid: https://permid.org

In this pythoncode, I will access OPenfigi and PermID APIs. Will use five companies traded on the 

In [97]:
#import necessary libraries
import requests
import json
import pandas as pd
import pprint
import os
import dotenv
from dotenv import load_dotenv
load_dotenv()


False

In [98]:
#get identifiers from Euronext Paris  
# Random list of 5 tickers on Euronext Paris for testing purposes

#ticks = ['ADS', 'BMW', 'FME', 'IFX', 'HEI'] #, 'DPW.DE', 'DTE.DE', 'FME.DE', 'FRE.DE', 'HEI.DE', 'HEN3.DE', 'IFX.DE', 'LIN.DE', 'MRK.DE', 'MUV2.DE', 'RWE.DE', 'SAP.DE', 'SIE.DE', 'VOW3.DE']
ticks = ['BNP', 'MC', 'ORA', 'SAN', 'VIE']


In [99]:
#Market Identifier Code(mic) for Euronext Paris is XPAR

jobs = [{'idType': 'TICKER','idValue': tick, 'micCode': 'XPAR'} 
        for tick in ticks]

pprint.pprint(jobs)

[{'idType': 'TICKER', 'idValue': 'BNP', 'micCode': 'XPAR'},
 {'idType': 'TICKER', 'idValue': 'MC', 'micCode': 'XPAR'},
 {'idType': 'TICKER', 'idValue': 'ORA', 'micCode': 'XPAR'},
 {'idType': 'TICKER', 'idValue': 'SAN', 'micCode': 'XPAR'},
 {'idType': 'TICKER', 'idValue': 'VIE', 'micCode': 'XPAR'}]


In [100]:
#API request to OpenFIGI. OpenFIGI documentation: https://www.openfigi.com/api

openfigi_url = 'https://api.openfigi.com/v3/mapping'
headers = {'Content-Type': 'application/json'}
response = requests.post(openfigi_url, headers=headers, data=json.dumps(jobs))
if response.status_code != 200:
    raise Exception(f"Error: {response.status_code}, {response.text}")
result = response.json()

In [101]:
#print the result
pprint.pprint(result)

[{'data': [{'compositeFIGI': 'BBG000BBXJ04',
            'exchCode': 'FP',
            'figi': 'BBG000BBXJ95',
            'marketSector': 'Equity',
            'name': 'BNP PARIBAS',
            'securityDescription': 'BNP',
            'securityType': 'Common Stock',
            'securityType2': 'Common Stock',
            'shareClassFIGI': 'BBG001S67SB2',
            'ticker': 'BNP'}]},
 {'data': [{'compositeFIGI': 'BBG000BC7PK5',
            'exchCode': 'FP',
            'figi': 'BBG000BC7Q05',
            'marketSector': 'Equity',
            'name': 'LVMH MOET HENNESSY LOUIS VUI',
            'securityDescription': 'MC',
            'securityType': 'Common Stock',
            'securityType2': 'Common Stock',
            'shareClassFIGI': 'BBG001S641T5',
            'ticker': 'MC'}]},
 {'data': [{'compositeFIGI': 'BBG000DQSH27',
            'exchCode': 'FP',
            'figi': 'BBG000DQSHL6',
            'marketSector': 'Equity',
            'name': 'ORANGE',
            'securit

In [102]:
#just to show the dataframe manipulation steps clearly
just_dict = [d['data'][0] for d in result]
df_figi = pd.DataFrame(just_dict)
df_figi

,figi,name,ticker,exchCode,compositeFIGI,securityType,marketSector,shareClassFIGI,securityType2,securityDescription
0,BBG000BBXJ95,BNP PARIBAS,BNP,FP,BBG000BBXJ04,Common Stock,Equity,BBG001S67SB2,Common Stock,BNP
1,BBG000BC7Q05,LVMH MOET HENNESSY LOUIS VUI,MC,FP,BBG000BC7PK5,Common Stock,Equity,BBG001S641T5,Common Stock,MC
2,BBG000DQSHL6,ORANGE,ORA,FP,BBG000DQSH27,Common Stock,Equity,BBG001S8C6V8,Common Stock,ORA
3,BBG000BWBBP2,SANOFI,SAN,FP,BBG000BWBBF3,Common Stock,Equity,BBG001SCSQN7,Common Stock,SAN
4,BBG000CSHJP7,VEOLIA ENVIRONNEMENT,VIE,FP,BBG000CSHJ86,Common Stock,Equity,BBG001SFJRV9,Common Stock,VIE


In [103]:
#The dataframe contains more columns than needed. We will filter to keep only the
#columns of interest and set ticker as index
columns_of_interest = ['name', 'ticker', 'marketSector', 'exchCode', 'figi']
df_figi = df_figi[columns_of_interest].set_index('ticker')
df_figi

,name,marketSector,exchCode,figi
ticker,,,,
BNP,BNP PARIBAS,Equity,FP,BBG000BBXJ95
MC,LVMH MOET HENNESSY LOUIS VUI,Equity,FP,BBG000BC7Q05
ORA,ORANGE,Equity,FP,BBG000DQSHL6
SAN,SANOFI,Equity,FP,BBG000BWBBP2
VIE,VEOLIA ENVIRONNEMENT,Equity,FP,BBG000CSHJP7


Refinitv PermID section

In [104]:
access_token = os.getenv('API_KEY3') # get API key from environment variable

In [105]:
#API endpoint for Refinitiv PermID
url = "https://api-eit.refinitiv.com/permid/match"
headers = {
    'Content-Type': 'text/plain',
    'Accept': 'application/json',
    'X-AG-Access-Token': access_token,
    'x-openmatch-numberOfMatchesPerRecord': '1',
    'x-openmatch-dataType': 'Organization',
}

In [106]:
#As per the documentation for PermID, we need to format the text with ticker && MIC. ~
#The first line in the etxt field is the standard identifier. We use 'Ticker'
text_field = 'Standard Identifier\n'

for tick in ticks:
    text_field += f'TICKER:{tick}' +'&&MIC:' + 'XPAR\n'
print(text_field)

Standard Identifier
TICKER:BNP&&MIC:XPAR
TICKER:MC&&MIC:XPAR
TICKER:ORA&&MIC:XPAR
TICKER:SAN&&MIC:XPAR
TICKER:VIE&&MIC:XPAR



In [107]:
response = requests.post(url, headers=headers, data=text_field)
r = response.json()
#print(response.headers)
#print(r)

In [108]:
pprint.pprint(r['outputContentResponse'])

KeyError: 'outputContentResponse'

In [109]:
for company in r['outputContentResponse']:
    print(company['Match OrgName'] + ' --> ' + company['Match OpenPermID'])

KeyError: 'outputContentResponse'

In [14]:
#define a function to retrieve data from the urls
def permid_data(permid_url):
    permid_headers = {
        'Accept': 'text/turtle'
    }

    permid_params = {
        'format': 'json-ld',
        'access-token': access_token
    }

    #actual request
    permid_response = requests.get(permid_url, headers=headers,params=permid_params)

    #convrt the response to JSON
    permid_data = json.loads(permid_response.content)

    return permid_data

In [15]:
#create empty dictionary
permid_dict = {}

#loop thru all tickers and put the data in dictionary

for tick, i in zip(ticks, r['outputContentResponse']):
    #the PermidID url for the ticker from the reponse earlier
    permid_url = i['Match OpenPermID']

    #'use function efined above to download data'
    data = permid_data(permid_url)

    #Put desired data in dictionary for the ticker
    permid_dict[tick] = {
        'company': data['vcard:organization-name'],
        'IPO'    : data['hasIPODate'],
        'address': data['mdaas:HeadquartersAddress'],
        'website': data['hasURL'],
        'phone'  : data['tr-org:hasHeadquartersPhoneNumber'],
        'LEI'    : data['tr-org:hasLEI'],
        'permid' : data['tr-common:hasPermId'],
        'permid_url' : permid_url
    }

In [16]:
df_permid = pd.DataFrame.from_dict(permid_dict, orient='index')

df_permid

,company,IPO,address,website,phone,LEI,permid,permid_url
BNP,BNP Paribas SA,1993-10-18T04:00:00Z,16 Boulevard Des Italiens\nParis 9\nPARIS\nILE...,https://group.bnpparibas/,33140144546,R0MUWSFPU8MPRO8K5P83,8589934326,https://permid.org/1-8589934326
MC,LVMH Moet Hennessy Louis Vuitton SE,1985-01-07T05:00:00Z,"22, avenue Montaigne\nPARIS\nILE-DE-FRANCE\n75...",https://www.lvmh.com/fr,33144132222,IOG4E947OATN0KJYSD45,4295866862,https://permid.org/1-4295866862
ORA,Orange SA,1997-10-20T04:00:00Z,"111, quai du President Roosevelt\nCs 70222\nIs...",https://www.orange.com,33144442222,969500MCOONR8990S771,4295868416,https://permid.org/1-4295868416
SAN,Sanofi SA,2002-07-01T04:00:00Z,46 Avenue de la Grande Armee\nPARIS\nILE-DE-FR...,https://www.sanofi.com/,33153774000,549300E9PC51EN656011,4295868215,https://permid.org/1-4295868215
VIE,Veolia Environnement SA,2000-07-20T04:00:00Z,21 Rue La Boetie\nPARIS\nILE-DE-FRANCE\n75008\...,https://www.veolia.com/,33185577000,969500LENY69X51OOT31,4295867473,https://permid.org/1-4295867473


In [17]:
#df_final=df_figi.join(df_permid)

#df_final

This section below is a test code to try out the marginalia search tool. ## Not related to OPenFIGI/Permid ##

In [18]:
import requests

url = "https://api.marginalia.nu/{key}/search/{query}";

rsp = requests.get(url.format(key='public', query="Lake Vostok"));

if rsp.ok:
  data = rsp.json()
  print ("Query: ", data['query'])
  print ("License: ", data['license'])
  print ("")
  for result in data['results']:
      print (result['url'])
      print ("\t" + result['title'])
      print ("\t" + result['description'])
      print ("")
else:
    print ("Bad Status " + str(rsp.status_code))

Query:  Lake Vostok
License:  CC-BY-NC-SA 4.0

https://en.wikipedia.org/wiki/Lake_Vostok
	Lake Vostok
	Lake Vostok (Russian romanized: ozero Vostok is the largest of Antarctica's 675 known subglacial lakes. Lake Vostok is located at the southern Pole of Cold, beneath Russia's Vostok Station under the surface of the central East Antarctic Ice Sheet, which i

https://southpolestation.com/trivia/10s/lakevostok.html
	Lake Vostok Drilling Project
	There wasn't any news. Apparently Russia did not conduct any drilling-related activity at Vostok in 2015-16 the only thing they presented at the 2016 Antarctic Treaty meeting in Santiago, Chile was a report containing the results of laboratory experiments

http://www.greatdreams.com/blog-2012/dee-blog130.html
	Dee Finney's blog  February 7, 2012  page 130  WHAT'S UP 
WITH LAKE VOSTOK?
	I deal with interpreteting aeromagnetic imagery daily in my mineral exploration work here in OZ. The huge size and intensity of the above mentioned magnetic anomaly